# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [15]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN was not found in Colab Secrets."

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


### Contract

- **Unit of analysis:** One row represents one content item for one client on one report date.
- **Table:** `fact_content_daily_performance`, using the March 2026 partition for initial analysis.
- **Time window:** March 2026 (`2026-03-01` through `2026-03-31`), used as a mid-panel development window.
- **Decision / prediction target:** The lane is Refresh / Content Opportunity Scoring. The eventual task will rank content items by their likelihood of needing review or refresh based on future performance movement. The future outcome will be defined separately from current features so that future information does not leak into the feature set.
- **Deliberately excluded:** `client_hash_id` and `content_hash_id` will not be used as model features because they are identifiers, not meaningful predictive signals.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [16]:
import duckdb

con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB + Hugging Face connection is ready.")

DuckDB + Hugging Face connection is ready.


In [17]:
files_query = """
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
WHERE file LIKE '%fact_content_daily_performance%'
ORDER BY file
"""

files_df = con.execute(files_query).df()

files_df

,file
0,hf://datasets/FlyRank/internship-warehouse/fac...
1,hf://datasets/FlyRank/internship-warehouse/fac...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [18]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

schema_query = f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_path}')
"""

schema_df = con.execute(schema_query).df()

schema_df

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Feature fields

The initial feature set will contain no more than five fields:

1. `gsc_impressions` — number of Google Search Console impressions observed for the content item.
2. `gsc_clicks` — number of Google Search Console clicks observed.
3. `gsc_avg_position` — average search position observed in Google Search Console.
4. `ga4_sessions` — number of GA4 sessions observed.
5. `scroll_events` — recorded scroll events observed.

### Label

The final refresh/opportunity label is not taken directly from this daily table. It will be defined from future performance movement in later modeling work. This separation prevents future information from becoming a feature.

### Context fields

- `report_date` — identifies when the observation was measured.
- `month` — identifies the warehouse month.
- `gsc_data_available` — indicates whether usable GSC data is available.
- `ga4_data_available` — indicates whether usable GA4 data is available.
- `client_has_gsc` — indicates whether the client has GSC data.
- `client_has_ga4` — indicates whether the client has GA4 data.

### Excluded fields

- `client_hash_id` — identifier only; not a predictive feature.
- `content_hash_id` — identifier only; not a predictive feature.
- Future-period performance fields — excluded because they would not be available at the decision moment and could cause leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain verification

The expected grain is one row per client, content item, and report date. This query checks whether any combination occurs more than once.

In [19]:
grain_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_df = con.execute(grain_query).df()

print("Duplicate grain combinations found:", len(grain_df))
grain_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,client_hash_id,content_hash_id,report_date,row_count


### Query 2 — Row count and date span

This query measures the number of March 2026 observations and verifies the actual date range in the partition.

In [20]:
count_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
"""

count_window_df = con.execute(count_window_query).df()

count_window_df

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Data availability

This query checks how many March observations have usable GSC and GA4 data. The availability flags are tested explicitly with `IS TRUE`.

In [21]:
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{march_path}')
"""

availability_df = con.execute(availability_query).df()

availability_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Five-feature frame

The following five features are measured from the March 2026 development window. Each is intended to represent information available before or at the decision moment.

In [22]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet('{march_path}')
LIMIT 20
"""

feature_df = con.execute(feature_query).df()

feature_df

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


### Available when?

### Feature availability

| Feature | Meaning | Available when? |
|---|---|---|
| `gsc_impressions` | Search visibility observed for the content item. | Available at the decision moment because it represents already observed GSC impressions. |
| `gsc_clicks` | Search clicks observed for the content item. | Available at the decision moment because the clicks have already occurred. |
| `gsc_avg_position` | Average GSC search position observed for the content item. | Available at the decision moment because it represents already observed search performance. |
| `ga4_sessions` | GA4 sessions observed for the content item. | Available at the decision moment only when `ga4_data_available IS TRUE`. |
| `scroll_events` | Recorded engagement/scroll events. | Available at the decision moment because they represent already observed engagement events. |

Future-period performance values are not used as current features because they would not be available at the decision moment and could create leakage.

### Deliberate leakage test

To demonstrate the leakage problem, I intentionally use `gsc_clicks` as a proxy for a target derived from the same observed performance information. This is not an acceptable production feature because it allows information used to define the outcome to enter the feature set.

The purpose of this experiment is to demonstrate why label-derived information can produce an unrealistically strong result.

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_test_query = f"""
SELECT
    gsc_clicks,
    CASE
        WHEN gsc_clicks > 0 THEN 1
        ELSE 0
    END AS leaked_target
FROM read_parquet('{march_path}')
WHERE gsc_clicks IS NOT NULL
LIMIT 5000
"""

leak_test_df = con.execute(leak_test_query).df()

X_leak = leak_test_df[["gsc_clicks"]]
y_leak = leak_test_df["leaked_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.2,
    random_state=42,
    stratify=y_leak
)

leak_model = DecisionTreeClassifier(random_state=42)
leak_model.fit(X_train, y_train)

leak_predictions = leak_model.predict(X_test)
leak_accuracy = accuracy_score(y_test, leak_predictions)

print("Leaked-feature accuracy:", round(leak_accuracy, 4))

Leaked-feature accuracy: 1.0


### Leakage result

The deliberately leaked feature produces an artificially strong result because `leaked_target` is constructed directly from `gsc_clicks`, which is also used as the feature. This is not a valid prediction setup because the feature contains the information used to define the outcome.

This experiment demonstrates why label-derived or outcome-derived information must be removed before modeling.

The leaked target and leakage construction are therefore excluded from the final feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

- The March 2026 partition is a development window and does not by itself establish future performance movement.
- GSC and GA4 availability is not uniform, so observations without the relevant data cannot be interpreted as equivalent to true zero activity.
- The dataset is observational. It can show measured associations and support ranking or decision-making, but it cannot establish that refreshing a page will cause better Google performance.
- Client history depth can differ, so a single calendar window may not represent the same amount of historical information for every client.
- Future-period information must be kept separate from current features to avoid leakage.
- The analysis is therefore decision-support and directional rather than causal.

In [24]:
# Section 4 — Data limits: supporting checks

data_limits_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_rows,
    COUNT(DISTINCT client_hash_id) AS clients_observed
FROM read_parquet('{march_path}')
"""

data_limits_df = con.execute(data_limits_query).df()
data_limits_df

,total_rows,gsc_available_rows,ga4_available_rows,gsc_unavailable_rows,ga4_unavailable_rows,clients_observed
0,9841378,3611061,413966,6230317,9427412,55


In [25]:
# Check whether the March window contains multiple observation dates
window_check_query = f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS distinct_report_dates
FROM read_parquet('{march_path}')
"""

window_check_df = con.execute(window_check_query).df()
window_check_df

,first_date,last_date,distinct_report_dates
0,2026-03-01,2026-03-31,31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.